# 🎓 Day 1 · Session 3
# 03C · Structured Outputs & Prompt Patterns

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Generate predictable structured outputs
- Produce valid JSON responses
- Create Markdown tables automatically
- Design reusable prompt templates
- Understand why structured outputs are essential in AI applications

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from openai import OpenAI
import pandas as pd
import json

In [2]:
repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

env_path = repo_root / ".env"

print("Repository root:", repo_root)
print("Loading .env from:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(f"OPENAI_API_KEY not found in {env_path}")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

Repository root: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai
Loading .env from: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.env
Env exists: True
OpenAI client initialized successfully.


In [3]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.3,
               system_prompt="You are a helpful AI teaching assistant.",
               max_tokens=800):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# Why Structured Outputs?

Most beginners ask AI to generate paragraphs.

Real AI applications rarely want paragraphs.

Instead they need:

- JSON
- Tables
- Lists
- CSV
- XML
- SQL
- HTML

Machines consume structured data much better than free text.

# 1️⃣ Why Structured Outputs Matter

Free text is for humans. JSON/tables are for software.

In [4]:
student_text = "Priya Sharma, Roll 21CSB044, scored 87 in ML and 92 in DSA."
prompt = f"Extract student information from this text:\n{student_text}"
print(ask_openai(prompt))

Student Information:
- Name: Priya Sharma
- Roll Number: 21CSB044
- Scores: 
  - ML: 87
  - DSA: 92


# 2️⃣ Ask for JSON

In [5]:
json_prompt = f"""
Extract student information as valid JSON.

Required schema:
{{
  "name": "...",
  "roll_number": "...",
  "scores": {{
    "subject": marks
  }}
}}

Text:
{student_text}
"""
print(ask_openai(json_prompt))

```json
{
  "name": "Priya Sharma",
  "roll_number": "21CSB044",
  "scores": {
    "ML": 87,
    "DSA": 92
  }
}
```


# 3️⃣ JSON Mode

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "Return valid JSON only."},
        {"role": "user", "content": json_prompt}
    ]
)
json_text = response.choices[0].message.content
print(json_text)
data = json.loads(json_text)
data

{
  "name": "Priya Sharma",
  "roll_number": "21CSB044",
  "scores": {
    "ML": 87,
    "DSA": 92
  }
}


{'name': 'Priya Sharma',
 'roll_number': '21CSB044',
 'scores': {'ML': 87, 'DSA': 92}}

# 4️⃣ Markdown Table Output

In [7]:
table_prompt = """
Compare zero-shot, few-shot and chain-of-thought prompting.

Return as a markdown table with columns:
Technique, Best Used For, Example Use Case, Risk.
"""
print(ask_openai(table_prompt))

Here's a comparison of zero-shot, few-shot, and chain-of-thought prompting in a markdown table format:

| Technique          | Best Used For                          | Example Use Case                                | Risk                                          |
|--------------------|---------------------------------------|------------------------------------------------|-----------------------------------------------|
| Zero-shot prompting | Tasks where the model has general knowledge but no specific examples are provided | Asking a model to summarize a text without prior examples | The model may produce irrelevant or inaccurate responses due to lack of context |
| Few-shot prompting  | Tasks where the model benefits from a few examples to understand the desired output format | Providing a few examples of question-answer pairs to generate similar pairs | The model may overfit to the provided examples and fail to generalize to new inputs |
| Chain-of-thought prompting | Complex reas

# 5️⃣ Prompt Pattern: The Explainer

In [8]:
explainer_template = """
You are a teacher for {level} students studying {subject}.

Explain {concept} using:
1. A one-sentence definition
2. A real-world analogy from {domain}
3. One worked example
4. One common misconception to avoid
5. One check-your-understanding question

Keep it under {word_limit} words.
"""
prompt = explainer_template.format(
    level="second-year engineering",
    subject="machine learning",
    concept="gradient descent",
    domain="hill climbing",
    word_limit=180
)
print(ask_openai(prompt))

1. **Definition**: Gradient descent is an optimization algorithm used to minimize a function by iteratively moving towards the steepest descent direction defined by the negative gradient.

2. **Analogy**: Imagine you're at the top of a foggy mountain and want to find the lowest point. You can’t see far ahead, so you feel the ground around you and take a step in the direction that slopes down the most. You repeat this process until you can’t go any lower.

3. **Worked Example**: Suppose we have a simple quadratic function \( f(x) = (x - 3)^2 \). The gradient (derivative) is \( f'(x) = 2(x - 3) \). Starting at \( x = 0 \) with a learning rate of \( 0.1 \):
   - Compute the gradient: \( f'(0) = 2(0 - 3) = -6 \).
   - Update \( x \): \( x = 0 - 0.1 \times (-6) = 0.6 \).
   - Repeat until \( x \) approaches 3.

4. **Common Misconception**: Many believe that gradient descent always finds the global minimum; however, it can get stuck in local minima, especially in complex landscapes.

5. **Ch

# 6️⃣ Prompt Pattern: The Examiner

In [ ]:
examiner_prompt = """
Create 5 MCQ questions on supervised learning for second-year engineering students.

Difficulty: Medium

For each question include:
- Question
- Four options A-D
- Correct answer
- One-line explanation

Avoid repeated question patterns.
"""
print(ask_openai(examiner_prompt, max_tokens=1200))

# 7️⃣ Prompt Pattern: The Grader

In [ ]:
student_answer = """
Overfitting is when the model learns the training data too much.
It performs good on training data but bad on new data.
We can reduce it by using more data or regularization.
"""
grader_prompt = f"""
Grade this student answer on overfitting.

Rubric:
- Correctness: 40 marks
- Clarity: 30 marks
- Completeness: 30 marks

Return score, strengths, improvements and suggested improved answer.

Student answer:
{student_answer}
"""
print(ask_openai(grader_prompt, max_tokens=1000))

# 8️⃣ Prompt Pattern Library

In [ ]:
pd.DataFrame([
    {"Pattern": "Explainer", "Use": "Teach concepts", "Output": "Analogy + example + misconception"},
    {"Pattern": "Examiner", "Use": "Generate MCQs", "Output": "Questions + answer key"},
    {"Pattern": "Grader", "Use": "Evaluate submissions", "Output": "Score + feedback"},
    {"Pattern": "Socratic Tutor", "Use": "Guide students", "Output": "Hints and questions"},
    {"Pattern": "Research Assistant", "Use": "Summarize papers", "Output": "Key ideas + gaps"},
])

# 🧪 Lab

Create one structured-output prompt and one reusable prompt pattern for your subject.

# ✅ Summary

Structured outputs make LLMs usable in applications.